In [ ]:
import random
from datasets import load_dataset
from typing import Generator, Iterable
import json
import numpy as np

# 🧠 튜터의 노트: 이 데이터셋은 한국어 음성 합성(TTS)에 특화된 데이터입니다.
# 즉, 텍스트(text)를 입력하면 해당 텍스트에 맞는 음성 파일(audio)을 출력하는 것이 목표죠.
# 이 데이터를 분석하며 "어떤 텍스트가 어떤 감정/발음을 갖춰야 자연스러울까?"를 고민하는 것이 핵심입니다!

DATASET_NAME = "daje/korean-tts-training"
SAMPLE_COUNT = 10  # 너무 많은 샘플을 처리하면 시간이 오래 걸리므로, 10개만 탐험해 봅시다!

print("🌟 TTS 마스터가 되기 위한 데이터 탐험 스크립트를 시작합니다! 🌟")
print("=" * 70)

# --- 🛠️ 데이터 로딩 전략: 스트리밍 vs. 메모리 로딩 ---
# 데이터셋이 매우 크다면 전체를 메모리에 올리는 것은 위험합니다.
# 따라서, 스트리밍(streaming=True)을 시도하고, 실패할 경우 작은 샘플만 불러오겠습니다.

dataset = None
sample_data_list = []

try:
    # 1. 스트리밍 모드로 시도 (대용량 데이터셋에 최적)
    print("✅ 1단계: 스트리밍(Streaming) 모드로 데이터셋 로드 시도 중... (최대 10개 샘플)")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("   🎉 성공! 스트리밍 모드로 데이터 로드를 시작합니다.")

except Exception as e:
    # 2. 스트리밍 실패 시 (예: 로컬 환경 제약), 작은 배치를 로드합니다.
    print(f"🚨 스트리밍 로드 실패: {e} (걱정 마세요! 작은 샘플로 우회합니다.)")
    try:
        dataset = load_dataset(DATASET_NAME, split='train')
        print("   💡 일반 모드로 데이터셋을 다운로드하여 진행합니다.")
    except Exception as e_fallback:
        print(f"❌ 치명적인 오류 발생: 데이터셋 로드에 실패했습니다. {e_fallback}")
        print("스크립트를 종료합니다.")
        # 데이터셋이 로드되지 않았다면 여기서 종료
        exit()

# 💡 샘플 데이터 확보 패턴 (Iterator 사용)
# 메모리에 모두 올릴 필요 없이, 필요한 만큼만 꺼내와서 분석합니다.
print(f"\n✨ 총 {min(SAMPLE_COUNT, 10)}개의 샘플 데이터를 탐험합니다.")

# Dataset 객체에서 .take(N)을 사용한 이터레이터 패턴이 가장 안전하고 효율적입니다.
sample_iterator = dataset.take(SAMPLE_COUNT)
sample_data_list = list(sample_iterator)

print("------------------------------------------------------------------------")


# --- 🔎 Phase 1: 데이터 구조 파악 (EDA - Exploratory Data Analysis) ---
print("\n🧠 [Phase 1] 데이터 구조와 핵심 메타데이터 분석")

# 1. 어떤 카테고리들이 있는지 한눈에 살펴보기
print("\n👉 1-1. 주요 카테고리 목록 ('category' 컬럼) 확인:")
# set을 사용하여 중복을 제거하고 고유한 카테고리만 나열합니다.
unique_categories = set(item['category'] for item in sample_data_list)
print(f"   ✅ 발견된 고유 카테고리: {', '.join(sorted(list(unique_categories)))[:100]}...")


# 2. 텍스트와 감정 간의 관계 분석 (가장 재미있는 부분!)
print("\n👉 1-2. 샘플별 텍스트와 감정 카테고리 매칭 분석:")
for i, sample in enumerate(sample_data_list):
    text = sample['text']
    category = sample['category']
    
    # ✍️ 주석으로 데이터의 의미를 설명합니다.
    if "감정" in category or "억양" in category:
        print(f"  [{i+1}/{len(sample_data_list)}] 🎭 감정/억양 분석: '{text}' -> [Category: {category}]")
    elif "발음" in category:
        print(f"  [{i+1}/{len(sample_data_list)}] 🗣️ 발음 집중: '{text}' -> [Category: {category}] (주의 깊게 발음해 보세요!)")
    else:
        print(f"  [{i+1}/{len(sample_data_list)}] 📚 일반 분석: '{text[:30]}...' -> [Category: {category}]")


# --- 🤖 Phase 2: AI 실습 시뮬레이션 (창의적인 사용 예시) ---
print("\n" + "=" * 70)
print("🚀 [Phase 2] AI 시뮬레이션: LLM 기반의 '의도 탐지기' 만들기")
print("💡 목표: 텍스트가 주어졌을 때, 어떤 카테고리(감정/억양)가 적절할지 추론해 보는 시뮬레이션!")
print("------------------------------------------------------------------------")


def simulate_intent_detection(sample: dict) -> str:
    """
    데이터셋의 텍스트를 입력받아, 가장 적절한 '톤' 또는 '의도'를 추측하는 함수입니다.
    (실제 LLM처럼 동작하는 척!)
    """
    text = sample['text'].lower()
    category = sample['category']
    
    if "?" in text or "어떻게" in text or "궁금" in text:
        return "✅ [의문문] ❓ 정보 습득/질문 (Tone: Curious, Questioning)"
    elif "최고" in text or "너무" in text or "감사" in text:
        return "✅ [긍정 감정] 😊 매우 만족/기쁨 (Tone: Joyful, Enthusiastic)"
    elif "안 돼" in text or "실망" in text or "힘들어" in text:
        return "✅ [부정 감정] 😔 슬픔/불만족/실망 (Tone: Sad, Disappointed)"
    elif "비밀" in text or "꼭" in text:
        return "✅ [강조/지시] 🗣️ 강력한 메시지 전달 (Tone: Command, Assertive)"
    else:
        return f"✅ [기본] 📖 일반 서술 (Original Category: {category})"

print("\n✨ [실습] 다음 텍스트가 어떤 의도를 담고 있을까요? (LLM 역할 시뮬레이션)")
print("=" * 70)

for i, sample in enumerate(sample_data_list):
    text = sample['text']
    predicted_intent = simulate_intent_detection(sample)
    
    # 🌟 초보자를 위한 흥미 유발 코멘트
    print(f"\n--- (Example {i+1}) ---")
    print(f"🎤 입력 텍스트: '{text}'")
    print(f"🔮 LLM 예측 의도: {predicted_intent}")
    print(f"⚙️ 원본 데이터 카테고리: {sample['category']}")


# --- 🏁 결론 및 마무리 ---
print("\n\n========================================================================")
print("🎉 축하합니다! 데이터 분석 튜터링을 성공적으로 마쳤습니다! 🎉")
print("========================================================================")
print("✨ 배운 것 요약:")
print("1. 데이터 구조: TTS 데이터는 텍스트(Text)와 음성(Audio)이 쌍을 이룹니다.")
print("2. 핵심 분석: 단순히 텍스트를 읽는 것을 넘어, 카테고리('category')를 통해 텍스트의 *톤*과 *의도*를 파악하는 것이 AI의 핵심입니다.")
print("3. 실습 과정: 스트리밍 데이터 처리 및 패턴 매칭(LLM 시뮬레이션) 방법을 익혔습니다.")
print("\n💡 다음 단계 도전 과제:")
print("만약 이 데이터셋이 실제 사용 가능하다면, 위에서 추론한 '의도'를 바탕으로 실제 TTS 모델(예: Gemini TTS)의 API를 호출해 음성 파일을 생성해 볼 수 있습니다! 화이팅!")